In [41]:
from pyspark.sql import SparkSession, DataFrame, functions as F
import pyarrow.parquet as pq

In [42]:
table = pq.read_table("2025-01.parquet")

# Xem schema
print(table.schema)

# Ghi file mới với timestamp microseconds
pq.write_table(
    table,
    "2025-01-fixed.parquet",
    coerce_timestamps="us",
    allow_truncated_timestamps=True
)

id: string
speed: double
travel_time: int64
status: string
data_as_of: timestamp[ns]
link_id: string
link_points: string
encoded_poly_line: string
encoded_poly_line_lvls: string
owner: string
transcom_id: string
borough: string
link_name: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1646


In [43]:
spark = SparkSession.builder.appName("test link points").master("local[*]").getOrCreate()

In [44]:
df = spark.read.parquet("2025-01-fixed.parquet")

In [45]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- speed: double (nullable = true)
 |-- travel_time: long (nullable = true)
 |-- status: string (nullable = true)
 |-- data_as_of: timestamp_ntz (nullable = true)
 |-- link_id: string (nullable = true)
 |-- link_points: string (nullable = true)
 |-- encoded_poly_line: string (nullable = true)
 |-- encoded_poly_line_lvls: string (nullable = true)
 |-- owner: string (nullable = true)
 |-- transcom_id: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- link_name: string (nullable = true)



In [46]:
POINT_PATTERNS = r"^-?\d+\.\d+,-?\d+\.\d+$"
LINK_POINT_MAX_LEN = 255

In [47]:
def clean_link_points(df: DataFrame) -> DataFrame:
    # Chuẩn hóa chuỗi
    link_points = F.trim(F.col("link_points"))
    # Tách chuỗi thành danh sách các tọa độ
    points = F.split(link_points, r"\s+")
    # nếu độ dài của link_points >= 255 thì sẽ cắt cặp tọa độ cuối cùng 
    points = F.when(F.length("link_points") >= LINK_POINT_MAX_LEN,
                    F.slice(points, 1, F.size(points) - 1)
    ).otherwise(points)
    # Sau đó chỉ giữ các tọa độ đúng định dạng
    points = F.filter(points, lambda p: p.rlike(POINT_PATTERNS))
    new_df = df.withColumn("link_points",
                            F.when(F.size(points) > 0, F.array_join(points, " ")))
    return new_df


In [48]:
link_points1 = "40.6309004,-74.14569 40.63259,-74.145531 40.6343505,-74.145131 40.6352905,-74.144911 40.6395205,-74.143161 40.64452,-74.141111 40.64672,-74.140171 40.6474905,-74.13978 40.65243,-74.136261 40.6532706,-74.13549 40.65375,-74.13476 40.654011,-74.134041 40.654	"
link_points2 = "40.75719,-73.99724 40.76017,-74.00382 40.76185,-74.00725 40.76608,-74.01726"
link_points3 = "40.78802,-73.79003 40.7868604,-73.78838 40.7863506,-73.787771 40.7854504,-73.786831 40.78502,-73.786481 40.7848004,-73.78632 40.7844005,-73.78601 40.7837704,-73.785671 40.7831705,-73.78538 40.78264,-73.7852 40.7817,-73.78497 40.78116,-73.78489 40.78058,-7"
link_points4 = "40.7442206,-73.771661 40.7454306,-73.76907 40.745701,-73.76831 40.7465005,-73.7654 40.7476404,-73.76128 40.7480905,-73.76008 40.7488704,-73.75842 40.7492,-73.757551 40.74947,-73.75658 40.75032,-73.75319 40.7506906,-73.75108 40.7508404,-73.75023 40.75104,-"
link_points5 = "40.78795,-73.790191 40.78647,-73.78812 40.7862104,-73.787781 40.78587,-73.787381 40.7854106,-73.786931 40.7846506,-73.786261 40.7843806,-73.786071 40.78327,-73.78547 40.78266,-73.785271 40.7795604,-73.78463 40.77606,-73.78439 40.77373,-73.78426 40.77273,-"
link_points = [link_points1, link_points2, link_points3, link_points4, link_points5]

In [49]:
import re

In [50]:
for link_point in link_points:
    test = re.split(r"\s+", link_point)
    
    print(test)

['40.6309004,-74.14569', '40.63259,-74.145531', '40.6343505,-74.145131', '40.6352905,-74.144911', '40.6395205,-74.143161', '40.64452,-74.141111', '40.64672,-74.140171', '40.6474905,-74.13978', '40.65243,-74.136261', '40.6532706,-74.13549', '40.65375,-74.13476', '40.654011,-74.134041', '40.654', '']
['40.75719,-73.99724', '40.76017,-74.00382', '40.76185,-74.00725', '40.76608,-74.01726']
['40.78802,-73.79003', '40.7868604,-73.78838', '40.7863506,-73.787771', '40.7854504,-73.786831', '40.78502,-73.786481', '40.7848004,-73.78632', '40.7844005,-73.78601', '40.7837704,-73.785671', '40.7831705,-73.78538', '40.78264,-73.7852', '40.7817,-73.78497', '40.78116,-73.78489', '40.78058,-7']
['40.7442206,-73.771661', '40.7454306,-73.76907', '40.745701,-73.76831', '40.7465005,-73.7654', '40.7476404,-73.76128', '40.7480905,-73.76008', '40.7488704,-73.75842', '40.7492,-73.757551', '40.74947,-73.75658', '40.75032,-73.75319', '40.7506906,-73.75108', '40.7508404,-73.75023', '40.75104,-']
['40.78795,-73.7901

## Quan sát từng bước của `clean_link_points`
Chạy các cell import, tạo `spark`, khai báo hằng số, hàm và 5 chuỗi mẫu phía trên; sau đó chạy lần lượt các cell bên dưới. Phần này không cần đọc hay chuyển đổi file Parquet.

Mỗi bước thêm một cột để giữ lại dữ liệu trước đó. `show(truncate=False, vertical=True)` hiển thị đầy đủ từng bản ghi. Spark chỉ thực thi các biểu thức khi gọi action như `show()` hoặc `count()`.

Lưu ý: hàm gốc kiểm tra độ dài **chuỗi gốc**, và `F.trim` mặc định chỉ bỏ dấu cách ở hai đầu, không bỏ tab. Các cell này giữ nguyên hành vi của hàm.


In [51]:
# Dùng lại 5 chuỗi mẫu, thêm các trường hợp biên để quan sát rõ hơn.
cases = [(f"sample_{i}", value) for i, value in enumerate(link_points, start=1)]
cases += [
    ("null", None),
    ("empty", ""),
    ("spaces_only", "   "),
    ("tabs_at_edges", "\t40.1,-73.2\t"),
    ("invalid_token", "40.1,-73.2 bad 40.3,-73.4"),
]
# Một tọa độ hợp lệ về regex với tổng độ dài chính xác 254 / 255 / 256.
# Ở 255 và 256, hàm sẽ cắt cả tọa độ duy nhất dù nó khớp regex.
for length in (254, 255, 256):
    cases.append((f"length_{length}", "1." + "0" * (length - 6) + ",2.0"))

# Build sample rows with Spark expressions (no Python workers required).
sample_rows = F.array(*[
    F.struct(F.lit(case_id).alias("case_id"), F.lit(value).cast("string").alias("link_points"))
    for case_id, value in cases
])
test_df = spark.range(1).select(F.explode(sample_rows).alias("sample")).select("sample.*")
test_df.show(truncate=False, vertical=True)


-RECORD 0------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 case_id     | sample_1                                                                                                                                                                                                                                                          
 link_points | 40.6309004,-74.14569 40.63259,-74.145531 40.6343505,-74.145131 40.6352905,-74.144911 40.6395205,-74.143161 40.64452,-74.141111 40.64672,-74.140171 40.6474905,-74.13978 40.65243,-74.136261 40.6532706,-74.13549 40.65375,-74.13476 40.654011,-74.134041 40.654\t 
-RECORD 1-------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### 1. `link_points = F.trim(F.col("link_points"))`
So sánh chuỗi gốc và chuỗi sau trim. Cột JSON giúp nhìn thấy tab (`\t`) và dấu cách; hai cột độ dài cho thấy chính xác số ký tự bị bỏ.


In [52]:
step1 = (test_df
    .withColumn("trimmed", F.trim(F.col("link_points")))
    .withColumn("raw_length", F.length("link_points"))
    .withColumn("trimmed_length", F.length("trimmed"))
)
step1.select(
    "case_id", "link_points", "trimmed", "raw_length", "trimmed_length",
    F.to_json(F.struct("link_points", "trimmed")).alias("visible_whitespace"),
).show(truncate=False, vertical=True)


-RECORD 0-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 case_id            | sample_1                                                                                                                                                                                                                                                                                                                                                                                                                 

### 2. `points = F.split(link_points, r"\s+")`
Ở đây biến `link_points` trong hàm là biểu thức đã trim. Tab ở hai đầu vẫn có thể tạo phần tử chuỗi rỗng khi split.


In [53]:
step2 = step1.withColumn("points_split", F.split(F.col("trimmed"), r"\s+"))
step2.select(
    "case_id", "trimmed", "points_split",
    F.to_json("points_split").alias("points_split_json"),
    F.size("points_split").alias("count_after_split"),
).show(truncate=False, vertical=True)


-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 case_id           | sample_1                                                                                                                                                                                                                                                                                       
 trimmed           | 40.6309004,-74.14569 40.63259,-74.145531 40.6343505,-74.145131 40.6352905,-74.144911 40.6395205,-74.143161 40.64452,-74.141111 40.64672,-74.140171 40.6474905,-74.13978 40.65243,-74.136261 40.6532706,-74.13549 40.65375,-74.13476 40.654011,-74.134041 40.654\t                              
 points_split      | [40.6309004,-74.14569, 40.63259,-74.145531, 40.63435

### 3. Cắt phần tử cuối nếu độ dài chuỗi gốc >= 255
`slice` bắt đầu từ vị trí **1**. Cột `drop_last` cho biết điều kiện cắt; so sánh hai mảng để thấy phần tử bị bỏ (có thể là chuỗi rỗng do tab cuối dòng).


In [54]:
step3 = (step2
    .withColumn("drop_last", F.length("link_points") >= LINK_POINT_MAX_LEN)
    .withColumn("points_sliced", F.when(
        F.col("drop_last"),
        F.slice(F.col("points_split"), 1, F.size("points_split") - 1),
    ).otherwise(F.col("points_split")))
)
step3.select(
    "case_id", "raw_length", "drop_last", "points_split", "points_sliced",
).show(truncate=False, vertical=True)


-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 case_id       | sample_1                                                                                                                                                                                                                                                                        
 raw_length    | 256                                                                                                                                                                                                                                                                             
 drop_last     | true                                                                                                             

### 4. `points = F.filter(points, lambda p: p.rlike(POINT_PATTERNS))`
Chỉ giữ phần tử khớp regex. `rejected_points` hiển thị các phần tử bị loại ở riêng bước này.


In [55]:
step4 = (step3
    .withColumn("points_valid", F.filter(
        F.col("points_sliced"), lambda p: p.rlike(POINT_PATTERNS)))
    .withColumn("rejected_points", F.filter(
        F.col("points_sliced"), lambda p: ~p.rlike(POINT_PATTERNS)))
)
step4.select(
    "case_id", "points_sliced", "rejected_points", "points_valid",
).show(truncate=False, vertical=True)


-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 case_id         | sample_1                                                                                                                                                                                                                                                                      
 points_sliced   | [40.6309004,-74.14569, 40.63259,-74.145531, 40.6343505,-74.145131, 40.6352905,-74.144911, 40.6395205,-74.143161, 40.64452,-74.141111, 40.64672,-74.140171, 40.6474905,-74.13978, 40.65243,-74.136261, 40.6532706,-74.13549, 40.65375,-74.13476, 40.654011,-74.134041, 40.654] 
 rejected_points | [40.654]                                                                                                       

### 5. Ghép các tọa độ hợp lệ thành chuỗi
Nếu mảng không còn phần tử, `when` không có `otherwise` trả về `NULL`. Cột `cleaned_link_points` là giá trị sẽ ghi đè lên `link_points` trong hàm gốc.


In [56]:
step5 = step4.withColumn(
    "cleaned_link_points",
    F.when(F.size("points_valid") > 0, F.array_join("points_valid", " ")),
)
step5.select(
    "case_id", "link_points", "cleaned_link_points",
).show(truncate=False, vertical=True)


-RECORD 0--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 case_id             | sample_1                                                                                                                                                                                                                                                          
 link_points         | 40.6309004,-74.14569 40.63259,-74.145531 40.6343505,-74.145131 40.6352905,-74.144911 40.6395205,-74.143161 40.64452,-74.141111 40.64672,-74.140171 40.6474905,-74.13978 40.65243,-74.136261 40.6532706,-74.13549 40.65375,-74.13476 40.654011,-74.134041 40.654\t 
 cleaned_link_points | 40.6309004,-74.14569 40.63259,-74.145531 40.6343505,-74.145131 40.6352905,-74.144911 40.6395205,-74.143161 40.64452,-74.141111 40.6

### 6. Đối chiếu với hàm gốc
So sánh có xử lý `NULL`: kết quả chạy từng bước phải giống `clean_link_points(test_df)` ở mọi dòng.


In [57]:
function_result = clean_link_points(test_df).select(
    "case_id", F.col("link_points").alias("function_result"),
)
comparison = (step5.select("case_id", "cleaned_link_points")
    .join(function_result, "case_id")
    .withColumn("matches", F.col("cleaned_link_points").eqNullSafe(F.col("function_result")))
)
comparison.show(truncate=False, vertical=True)
assert comparison.filter(~F.col("matches")).count() == 0, "Kết quả khác hàm gốc"


-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 case_id             | sample_1                                                                                                                                                                                                                                                       
 cleaned_link_points | 40.6309004,-74.14569 40.63259,-74.145531 40.6343505,-74.145131 40.6352905,-74.144911 40.6395205,-74.143161 40.64452,-74.141111 40.64672,-74.140171 40.6474905,-74.13978 40.65243,-74.136261 40.6532706,-74.13549 40.65375,-74.13476 40.654011,-74.134041       
 function_result     | 40.6309004,-74.14569 40.63259,-74.145531 40.6343505,-74.145131 40.6352905,-74.144911 40.6395205,-74.143161 40.64452,-74.141111 40.64672,-74.

Để quan sát dữ liệu Parquet, thay cell tạo `test_df` bằng đoạn dưới rồi chạy lại từ bước 1 (sau khi đã đọc được `df`):
```python
test_df = (df.select("link_points").distinct().limit(20)
    .withColumn("case_id", F.monotonically_increasing_id().cast("string")))
```
